# Workshop Template — Apply PEFT to Your Own Model and Data

This notebook is a **fill-in-the-blanks template**. It runs end-to-end with sensible
defaults so you can verify the setup works, then swap in your own model, dataset and
adaptation method.

**Workflow:**

```
1. Config      ← set your choices once at the top
2. Model       ← load any ViT-style backbone (HF or custom)
3. Dataset     ← load any image classification dataset
4. PEFT method ← pick one from src/methods and configure it
5. Train       ← standard training loop from src/training
6. Evaluate    ← accuracy curve + trainable-param summary
```

Every cell marked `# ── USER SECTION ──` is meant to be edited.
Cells without that marker can be left as-is.

## 0. Setup

In [ ]:
# !pip install -q torch torchvision transformers peft matplotlib pandas tqdm
import sys, copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from transformers import ViTModel

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    sys.path.insert(0, str((ROOT / "..").resolve()))
elif (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT.resolve()))

from src.methods.linear_probe import LinearProbeModel
from src.methods.adapters import AdapterHeadClassifier
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.lora import LoRAClassifier
from src.methods.bitfit import BitFitClassifier
from src.methods.partial_ft import PartialFineTuneClassifier
from src.training import count_trainable_parameters, freeze_module, train_model, evaluate

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

---
## 1. Configuration

**Edit this cell.** Everything downstream reads from these variables.

In [ ]:
# ── USER SECTION ──────────────────────────────────────────────────────────────

# --- Model ---
MODEL_NAME = "WinKawaks/vit-small-patch16-224"   # any HF ViT-style model
IMG_SIZE   = 224                                  # must match model

# --- Dataset ---
# Set USE_SYNTHETIC = False and fill in the dataset section (Section 3) to use real data.
USE_SYNTHETIC = True
N_CLASSES     = 10
BATCH_SIZE    = 16

# --- Adaptation method ---
# Choose one of:
#   "linear_probe" | "bitfit" | "visual_prompt" | "lora" | "adapter" | "partial_ft"
METHOD = "lora"

# --- Method hyperparameters (only the selected method's params are used) ---
LORA_RANK          = 8
LORA_TARGET        = ["query", "value"]   # layer name suffixes to apply LoRA to
ADAPTER_BOTTLENECK = 32
PROMPT_SIZE        = 32                   # prompt patch side length (pixels)
PARTIAL_N_BLOCKS   = 1                    # how many last transformer blocks to unfreeze

# --- Training ---
EPOCHS = 10
LR     = 3e-3
# ── END USER SECTION ──────────────────────────────────────────────────────────

print("Method    :", METHOD)
print("Model     :", MODEL_NAME)
print("Synthetic :", USE_SYNTHETIC)
print("Classes   :", N_CLASSES)

---
## 2. Load the Backbone

The default loads a HuggingFace ViT and wraps it so it returns the CLS token.

**To swap in your own model:**
- Define any `nn.Module` that takes `(B, 3, H, W)` images and returns `(B, D)` features.
- Set `backbone.feature_dim = D` so the PEFT wrappers know the feature size.

In [ ]:
# ── USER SECTION (optional — swap in your own backbone) ──────────────────────

class HFViTBackbone(nn.Module):
    """Thin wrapper around a HuggingFace ViTModel that returns the CLS token."""
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        self.feature_dim = self.vit.config.hidden_size

    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]

# --- Replace the line below to use a different backbone ---
backbone = HFViTBackbone().to(device)
# ── END USER SECTION ──────────────────────────────────────────────────────────

freeze_module(backbone)
D     = backbone.feature_dim
total = sum(p.numel() for p in backbone.parameters())
print(f"Backbone     : {MODEL_NAME}")
print(f"Feature dim  : {D}")
print(f"Total params : {total:,}")

---
## 3. Load the Dataset

**Default (synthetic):** random images with labels derived from the frozen backbone —
useful for verifying the pipeline without downloading anything.

**To use a real dataset**, set `USE_SYNTHETIC = False` in the config cell and fill
in the `else` branch below. Any `torch.utils.data.Dataset` that returns `(image, label)`
tensors works. Example stubs are provided for CIFAR-10 and `ImageFolder`.

In [ ]:
if USE_SYNTHETIC:
    # ── Synthetic data — no changes needed ────────────────────────────────────
    N = 200
    X = torch.randn(N, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        H = backbone(X[:64].to(device)).cpu()      # feature batch to build labels
        # extend if needed
        chunks = [backbone(X[i:i+64].to(device)).cpu() for i in range(0, N, 64)]
        H = torch.cat(chunks)
    W_label = torch.randn(D, N_CLASSES)
    y = (H @ W_label).argmax(dim=-1)

    split = int(0.8 * N)
    train_ds = TensorDataset(X[:split],  y[:split])
    val_ds   = TensorDataset(X[split:],  y[split:])

else:
    # ── USER SECTION — replace with your real dataset ─────────────────────────
    #
    # Option A: torchvision CIFAR-10
    # from torchvision import datasets, transforms
    # tfm = transforms.Compose([
    #     transforms.Resize((IMG_SIZE, IMG_SIZE)),
    #     transforms.ToTensor(),
    #     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    # ])
    # train_ds = datasets.CIFAR10("data/", train=True,  download=True, transform=tfm)
    # val_ds   = datasets.CIFAR10("data/", train=False, download=True, transform=tfm)
    #
    # Option B: folder of images  (one sub-folder per class)
    # from torchvision.datasets import ImageFolder
    # train_ds = ImageFolder("data/train", transform=tfm)
    # val_ds   = ImageFolder("data/val",   transform=tfm)
    #
    raise NotImplementedError("Fill in your dataset above and remove this line.")
    # ── END USER SECTION ──────────────────────────────────────────────────────

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

print(f"Train samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}")
print(f"Classes       : {N_CLASSES}")

---
## 4. Build the Adapted Model

The factory below reads `METHOD` from the config cell and instantiates the right wrapper
from `src/methods/`. You can also call a wrapper directly if you want fine-grained control.

In [ ]:
def build_model(method: str, backbone: nn.Module, D: int, K: int) -> nn.Module:
    """Instantiate the chosen PEFT wrapper around a deep-copy of the backbone."""
    bb = copy.deepcopy(backbone)

    if method == "linear_probe":
        return LinearProbeModel(bb, D, K)

    if method == "bitfit":
        return BitFitClassifier(bb, D, K)

    if method == "visual_prompt":
        return PromptTunedClassifier(bb, D, K, prompt_size=PROMPT_SIZE)

    if method == "lora":
        return LoRAClassifier(bb, D, K,
                              target_modules=LORA_TARGET, rank=LORA_RANK)

    if method == "adapter":
        return AdapterHeadClassifier(bb, D, K, bottleneck_dim=ADAPTER_BOTTLENECK)

    if method == "partial_ft":
        blocks = list(bb.vit.encoder.layer)   # adjust path for custom backbones
        to_unfreeze = blocks[-PARTIAL_N_BLOCKS:]
        return PartialFineTuneClassifier(bb, D, K, modules_to_unfreeze=to_unfreeze)

    raise ValueError(f"Unknown method '{method}'. "
                     "Choose from: linear_probe, bitfit, visual_prompt, "
                     "lora, adapter, partial_ft")


model = build_model(METHOD, backbone, D, N_CLASSES).to(device)

total_p = sum(p.numel() for p in model.parameters())
train_p = count_trainable_parameters(model)
print(f"Method            : {METHOD}")
print(f"Trainable params  : {train_p:,}")
print(f"Total params      : {total_p:,}")
print(f"% of total        : {100*train_p/total_p:.3f}%")
print()
print("Trainable tensors:")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name:55s} {list(p.shape)}")

---
## 5. Train

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-2
)

history = train_model(model, train_loader, val_loader,
                      optimizer, epochs=EPOCHS, device=device)

print(f"\nFinal val accuracy : {history.val_acc[-1]:.3f}")

---
## 6. Evaluate and Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(history.train_loss, label="train loss")
axes[0].plot(history.val_loss,   label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title(f"{METHOD} — loss")
axes[0].legend()

axes[1].plot(history.val_acc, color="steelblue")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val accuracy")
axes[1].set_title(f"{METHOD} — validation accuracy")
axes[1].set_ylim(0, 1)

plt.suptitle(f"Training results  |  method={METHOD}  |  trainable={train_p:,} params",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. (Optional) Compare Multiple Methods

Run this section to train all six methods on the same data and compare their
final validation accuracy against trainable parameter count.

In [ ]:
ALL_METHODS = ["linear_probe", "bitfit", "visual_prompt", "lora", "adapter", "partial_ft"]

results = []
for m_name in ALL_METHODS:
    print(f"Training {m_name} ...", end=" ", flush=True)
    m = build_model(m_name, backbone, D, N_CLASSES).to(device)
    opt = torch.optim.AdamW(
        [p for p in m.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2
    )
    hist = train_model(m, train_loader, val_loader, opt, epochs=EPOCHS, device=device)
    tr = count_trainable_parameters(m)
    results.append({
        "method":      m_name,
        "trainable":   tr,
        "val_acc":     hist.val_acc[-1],
        "history":     hist,
    })
    print(f"val_acc={hist.val_acc[-1]:.3f}  trainable={tr:,}")

df_cmp = pd.DataFrame([{k: v for k, v in r.items() if k != "history"}
                        for r in results]).set_index("method")
print()
print(df_cmp.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy curves
for r in results:
    axes[0].plot(r["history"].val_acc, label=r["method"])
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val accuracy")
axes[0].set_title("Validation accuracy — all methods")
axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1)

# Accuracy vs trainable params
axes[1].scatter(df_cmp["trainable"], df_cmp["val_acc"], s=80, zorder=3)
for method, row in df_cmp.iterrows():
    axes[1].annotate(method, (row["trainable"], row["val_acc"]),
                     textcoords="offset points", xytext=(5, 3), fontsize=8)
axes[1].set_xscale("log")
axes[1].set_xlabel("Trainable parameters (log scale)")
axes[1].set_ylabel("Final val accuracy")
axes[1].set_title("Accuracy vs parameter efficiency")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()